In [15]:
import geopandas as gpd
import os
import re
import pandas as pd
import folium
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
property_full = gpd.read_file('output/property.gpkg', engine='pyogrio')
canopies = gpd.read_file('output/property_isodistance_canopies.gpkg')
access_point_og = gpd.read_file('output/property_accesspoints.gpkg')

In [5]:
property = property_full[['geometry', 'property_id']]

In [6]:
REACHES = [25, 50, 75, 100, 150, 200, 250, 300, 350, 400]

access_point = access_point_og.rename(columns= {'geometry': 'access_point'})

property = property.merge(
    access_point[['property_id', 'access_point']],
    on='property_id',
    how='left'
)

for distance in REACHES:
  path = f'output/property_reach_{distance}m.gpkg'
  reach_col = f'reach_{distance}m'
  
  if os.path.exists(path):
      reach_gdf = gpd.read_file(path, engine='pyogrio')
      reach_gdf = reach_gdf.rename(columns={'geometry': reach_col})
      
      property = property.merge(
          reach_gdf[['property_id', reach_col]],    
          on='property_id',
          how='left'
      )

# calculate reach lengths

In [7]:
for col in property.columns:
    if col.startswith("reach_"):
        dist = re.search(r'_(\d+)m', col).group(1)
        property[f'reach_len_{dist}m'] = property[col].length

# plotting

In [8]:
sample = property.sample(n=10, random_state=42)

In [9]:
REACH = 200
reach_col = f"reach_{REACH}m"
reach_len_col = f"reach_len_{REACH}m"

sample_wgs = sample.to_crs(4326).copy()
sample_wgs[reach_col] = gpd.GeoSeries(sample[reach_col], crs=sample.crs).to_crs(4326)
sample_wgs['access_point'] = gpd.GeoSeries(sample['access_point'], crs=sample.crs).to_crs(4326)

m = folium.Map(
  location = [sample_wgs.geometry.y.mean(), sample_wgs.geometry.x.mean()],
  zoom_start=14,
  tiles='CartoDB Positron'
)

for idx, row in sample_wgs.iterrows():
  prop = row['geometry']
  access = row['access_point']
  reach = row[reach_col]
  reach_length = row[reach_len_col]
  
  folium.CircleMarker(
    [prop.y, prop.x],
    radius=7,
    color='red',
    fill=True,
    fill_opacity=1,
    popup=f"Property {idx}"
  ).add_to(m)
  
  folium.CircleMarker(
    [access.y, access.x],
    radius=5,
    color='blue',
    fill=True,
    fill_opacity=0.8,
    popup=f"Access {idx}"
  ).add_to(m)
  
  folium.GeoJson(
        reach,
        style_function=lambda x: {
            "color": "red",
            "weight": 3,
            "opacity": 0.8
        },
        tooltip=f"{round(reach_length, 2)}m"
    ).add_to(m)

m

# normalization

In [10]:
property.columns

Index(['geometry', 'property_id', 'access_point', 'reach_25m', 'reach_50m',
       'reach_75m', 'reach_100m', 'reach_150m', 'reach_200m', 'reach_250m',
       'reach_300m', 'reach_350m', 'reach_400m', 'reach_len_25m',
       'reach_len_50m', 'reach_len_75m', 'reach_len_100m', 'reach_len_150m',
       'reach_len_200m', 'reach_len_250m', 'reach_len_300m', 'reach_len_350m',
       'reach_len_400m'],
      dtype='object')

In [11]:
canopies.columns

Index(['property_id', 'canopy_50m', 'canopy_100m', 'canopy_150m',
       'canopy_200m', 'canopy_250m', 'canopy_300m', 'canopy_350m',
       'canopy_400m', 'canopy_25m', 'canopy_75m', 'geometry'],
      dtype='object')

In [12]:
df = property.merge(
  canopies.drop(columns='geometry'),
  on='property_id',
  how='left'
)

In [13]:
df.columns

Index(['geometry', 'property_id', 'access_point', 'reach_25m', 'reach_50m',
       'reach_75m', 'reach_100m', 'reach_150m', 'reach_200m', 'reach_250m',
       'reach_300m', 'reach_350m', 'reach_400m', 'reach_len_25m',
       'reach_len_50m', 'reach_len_75m', 'reach_len_100m', 'reach_len_150m',
       'reach_len_200m', 'reach_len_250m', 'reach_len_300m', 'reach_len_350m',
       'reach_len_400m', 'canopy_50m', 'canopy_100m', 'canopy_150m',
       'canopy_200m', 'canopy_250m', 'canopy_300m', 'canopy_350m',
       'canopy_400m', 'canopy_25m', 'canopy_75m'],
      dtype='object')

In [16]:
canopy_norm = pd.DataFrame()
canopy_norm["property_id"] = df["property_id"]

In [18]:
denom_0_25 = df["reach_len_25m"]

canopy_norm["canopy_0_25"] = np.where(
    denom_0_25 > 0,
    df["canopy_25m"] / denom_0_25,
    np.nan
)

In [19]:
REACHES = [25, 50, 75, 100, 150, 200, 250, 300, 350, 400]

for r0, r1 in zip(REACHES[:-1], REACHES[1:]):

    c_hi = f"canopy_{r1}m"
    c_lo = f"canopy_{r0}m"
    l_hi = f"reach_len_{r1}m"
    l_lo = f"reach_len_{r0}m"

    out = f"canopy_{r0}_{r1}"

    denom = df[l_hi] - df[l_lo]

    canopy_norm[out] = np.where(
        denom > 0,
        (df[c_hi] - df[c_lo]) / denom,
        np.nan
    )


In [20]:
canopy_norm.describe()

,property_id,canopy_0_25,canopy_25_50,canopy_50_75,canopy_75_100,canopy_100_150,canopy_150_200,canopy_200_250,canopy_250_300,canopy_300_350,canopy_350_400
count,12317.000000,12317.000000,1.231700e+04,12317.000000,1.231700e+04,1.231700e+04,1.231700e+04,1.231700e+04,1.231700e+04,1.231700e+04,1.231700e+04
mean,6158.000000,2.119602,1.786169e+00,1.894364,1.980501e+00,2.007204e+00,2.026020e+00,2.101022e+00,2.124483e+00,2.165976e+00,2.137694e+00
std,3555.755967,2.851310,2.465311e+00,2.655104,2.890055e+00,2.689279e+00,2.643388e+00,2.921856e+00,2.946204e+00,3.291011e+00,2.970155e+00
min,0.000000,0.000000,-1.538690e-07,-0.000001,-3.875041e-07,-3.653130e-08,-3.205424e-08,-3.675312e-07,-1.644924e-07,-1.335529e-07,-1.636569e-09
25%,3079.000000,0.346900,3.173622e-01,0.344388,3.788775e-01,4.925253e-01,5.433796e-01,5.686005e-01,5.914936e-01,6.210627e-01,6.342008e-01
50%,6158.000000,1.189489,1.009563e+00,1.082701,1.128049e+00,1.216396e+00,1.255875e+00,1.291008e+00,1.320254e+00,1.333173e+00,1.348095e+00
75%,9237.000000,2.720784,2.281399e+00,2.414216,2.464055e+00,2.466045e+00,2.524246e+00,2.576752e+00,2.564389e+00,2.553422e+00,2.549077e+00
max,12316.000000,33.643405,6.125709e+01,62.098537,5.973212e+01,7.079309e+01,4.666827e+01,6.032371e+01,7.040305e+01,7.061111e+01,5.259070e+01


In [21]:
canopy_norm.head()

,property_id,canopy_0_25,canopy_25_50,canopy_50_75,canopy_75_100,canopy_100_150,canopy_150_200,canopy_200_250,canopy_250_300,canopy_300_350,canopy_350_400
0,0,0.289328,0.226332,0.908324,1.167321,0.241147,1.195941,1.383674,1.323826,1.355033,1.553575
1,1,0.551338,0.058572,0.161087,0.763813,1.266081,0.760155,2.158955,1.275698,1.672682,1.085619
2,2,0.651656,1.187614,0.306129,0.317109,0.569719,0.291474,0.666274,1.336082,1.825927,1.057311
3,3,0.000000,0.609909,0.030354,1.526245,0.943086,0.845029,2.260939,1.043788,1.702022,1.228919
4,4,0.185263,0.674242,1.192428,1.020600,0.129862,0.565455,1.019686,1.992117,1.058797,1.448678


In [ ]:
out_path = 'output/property_canopy_normalized.csv'

# canopy_norm.to_csv(out_path, index=False)